# 수치해석 02. 최소제곱과 QR

## 학습 목표
- 핵심 정의와 정리를 자신의 말로 설명한다.
- 기본 개념 예제를 손계산으로 확인한다.
- Python 코드와 시각화를 통해 직관을 검산한다.

## 핵심 개념
- 정사영
- 최소제곱
- 잔차
- 모델 적합

## 기본 개념 예제
노이즈 자료에 직선을 맞추고 잔차를 분석한다.

## 실제 응용 예제
실험 측정값에서 물리 상수를 추정한다.

## 이론 정리

### 정의와 관점
- 벡터공간은 덧셈과 스칼라배가 닫혀 있는 추상적 좌표 공간이다.
- 기저는 모든 벡터를 유일하게 표현하게 하는 최소 생성 집합이다.
- 행렬은 선형변환의 좌표 표현이고, 행렬곱은 변환의 합성이다.
- 고유값과 고유벡터는 변환이 방향을 보존하면서 크기만 바꾸는 특별한 축을 나타낸다.

### 핵심 명제와 정리
- 랭크-널리티 정리는 해공간의 자유도와 변환 이미지의 차원을 보존 관계로 묶는다.
- 스펙트럼 정리는 대칭행렬을 직교기저에서 대각화해 해석을 단순화한다.
- 최소제곱은 관측 벡터를 열공간에 정사영하는 문제로 해석된다.
- SVD는 임의 행렬을 회전, 축척, 회전의 조합으로 분해해 저차원 근사를 가능하게 한다.

### 계산과 학습 절차
- 문제를 벡터, 행렬, 부분공간, 선형변환 중 어느 언어로 보는지 먼저 정한다.
- 해공간 문제는 행렬의 랭크, 영공간, 열공간을 함께 확인한다.
- 고유값 문제는 반복 적용의 장기 거동, 안정성, 축 방향 해석과 연결한다.
- 데이터 문제에서는 정사영, 잔차, 조건수, 차원축소가 핵심 진단 도구다.

### 자주 생기는 오해
- 행렬이 정사각형이어도 항상 역행렬이 있는 것은 아니며, 조건수가 크면 수치적으로 불안정하다.
- 고유벡터가 항상 충분히 많지는 않으므로 대각화 가능성과 고유값 존재를 구분해야 한다.
- 상관이 큰 열을 가진 설계행렬은 최소제곱 계수 해석을 불안정하게 만든다.

### 증명으로 연결하기
- 선형대수 증명은 닫힘, 생성, 독립성, 차원 논증을 반복적으로 사용한다.
- 분해정리는 부분공간의 직교성 또는 불변 부분공간을 찾아 문제를 작은 블록으로 나눈다.
- 최적성 증명은 잔차가 열공간에 직교한다는 정사영 조건에서 출발한다.

### 이 챕터에서 꼭 확인할 질문
- 기본 개념 예제 "노이즈 자료에 직선을 맞추고 잔차를 분석한다."에서 실제로 사용한 정의는 무엇인가?
- 응용 예제 "실험 측정값에서 물리 상수를 추정한다."에서 어떤 가정이 현실을 단순화하고 있는가?
- 코드가 연속 대상을 이산화한다면, 격자나 표본 수를 바꾸어도 결론이 유지되는가?
- 손계산 가능한 작은 사례와 노트북 결과가 같은 결론을 주는가?

## 0. 실행 준비

아래 셀은 프로젝트 루트의 `common/math_viz.py`를 찾아서 현재 챕터의 출력 폴더를 자동으로 설정합니다. Jupyter Lab을 프로젝트 루트에서 열면 가장 안정적으로 동작합니다.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display


CHAPTER_RELATIVE_DIR = Path("3학년_순수수학과_응용수학의_분화/08_수치해석/ch02_최소제곱과_QR")


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "common" / "math_viz.py").exists():
            return candidate
    raise RuntimeError("common/math_viz.py를 찾지 못했습니다. Jupyter Lab을 프로젝트 루트에서 열어 주세요.")


ROOT = find_project_root(Path.cwd().resolve())
NOTEBOOK_DIR = ROOT / CHAPTER_RELATIVE_DIR
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
sys.path.insert(0, str(ROOT / "common"))

from math_viz import PROFILES, run_profile

PROFILE = "least_squares"
TITLE = "수치해석 - 최소제곱과 QR"
CONCEPT_EXAMPLE = "노이즈 자료에 직선을 맞추고 잔차를 분석한다."
APPLICATION_EXAMPLE = "실험 측정값에서 물리 상수를 추정한다."

print("project root:", ROOT)
print("chapter dir:", NOTEBOOK_DIR)
print("profile:", PROFILE)

## 1. 이번 챕터의 시각화 코드 읽기

먼저 실제로 실행될 함수를 확인합니다. 코드를 읽으면서 입력값, 이산화 방식, 그래프가 의미하는 수학적 대상을 표시해 보세요.

In [ ]:
import inspect

print(inspect.getsource(PROFILES[PROFILE]))

## 2. 실행하고 결과 확인하기

아래 셀을 실행하면 `outputs/visualization.png`가 생성되고, 노트북 안에도 바로 표시됩니다.

In [ ]:
run_profile(
    profile=PROFILE,
    title=TITLE,
    concept=CONCEPT_EXAMPLE,
    application=APPLICATION_EXAMPLE,
    output_dir=OUTPUT_DIR,
)

display(Image(filename=str(OUTPUT_DIR / "visualization.png")))

## 3. 변형 실험

- 표본 수, 격자 크기, 초기값, 학습률, 경계조건 중 하나를 바꿔 보세요.
- 그림이 안정적으로 유지되는 범위와 결론이 바뀌는 범위를 나누어 적어 보세요.
- 손계산 가능한 작은 예제를 만들어 코드 결과와 비교해 보세요.

In [ ]:
# 여기에 자신만의 변형 실험을 작성하세요.
# 예: common/math_viz.py에서 위에 출력된 함수의 파라미터를 복사해 와서
#     표본 수, 구간, 초기값 등을 바꾼 뒤 다시 그려 볼 수 있습니다.
